# 07 · Grad-CAM:模型到底在看哪裡?

CNN 說「這是一隻狗」,信心 98%。但它是**看了狗的臉**才這樣判斷,還是**看了背景的草地**剛好猜對?分不清楚,你就不敢信任它。

**Grad-CAM** 是最常用的 CNN 可解釋性工具:

> 它利用最後一層卷積的梯度,算出影像上**哪些區域對「這個預測」貢獻最大**,畫成一張熱力圖疊在原圖上。紅的地方,就是模型「盯著看」的地方。

這呼應 `ml/scikit-learn` 軌道教過的 SHAP——同樣是「打開黑盒子、解釋模型為什麼這樣判斷」。

## 1. 安裝

In [ ]:
%pip install -q -U torchvision

In [ ]:
%pip install -q -U grad-cam

In [ ]:
import requests
from PIL import Image


def load_image(url):
    """從網址載一張圖,回傳 RGB 的 PIL.Image。"""
    return Image.open(requests.get(url, stream=True).raw).convert("RGB")

## 2. 載入預訓練 ResNet-50,先做一次預測

用 torchvision 內建的權重,連 ImageNet 的類別名稱一起拿到。

In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]

# 經典的貓+狗範例圖(grad-cam 套件自帶)
img = load_image("https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/both.png")
x = preprocess(img).unsqueeze(0)
with torch.no_grad():
    probs = model(x).softmax(1)[0]
top = int(probs.argmax())
print(f"預測:{categories[top]}  信心 {probs[top]:.2f}")

## 3. 算 Grad-CAM 熱力圖

目標層選**最後一個卷積區塊**(`layer4[-1]`)——它的特徵圖最具語意。熱力圖疊回原圖,紅 = 模型最看重的區域。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

cam = GradCAM(model=model, target_layers=[model.layer4[-1]])
grayscale = cam(input_tensor=x, targets=[ClassifierOutputTarget(top)])[0]

rgb = np.array(img.resize((224, 224))).astype("float32") / 255.0
vis = show_cam_on_image(rgb, grayscale, use_rgb=True)

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(rgb); ax[0].set_title("原圖"); ax[0].axis("off")
ax[1].imshow(vis); ax[1].set_title(f"Grad-CAM: {categories[top]}"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 4. 換一個目標類別,看它改看哪裡

同一張貓+狗的圖,叫它解釋「另一個類別」,熱力圖會移到對應的動物身上——證明模型確實**分得清不同物件在圖的哪裡**。

In [ ]:
# 取信心第二高的類別,看熱區是否移動
second = int(probs.argsort(descending=True)[1])
g2 = cam(input_tensor=x, targets=[ClassifierOutputTarget(second)])[0]
vis2 = show_cam_on_image(rgb, g2, use_rgb=True)
plt.figure(figsize=(4.5, 4.5))
plt.imshow(vis2); plt.title(f"Grad-CAM: {categories[second]}"); plt.axis("off"); plt.show()

## 小結

- **Grad-CAM 打開 CNN 黑盒子**:用梯度算出哪些區域對某預測貢獻最大,畫成熱力圖。
- 能抓出「**看對地方**」還是「**靠背景瞎猜**」——除錯與建立信任的利器。
- 與 sklearn 軌道的 **SHAP** 一脈相承:可解釋性是把模型用在真實場景的必修課。

下一課(壓軸):把整條軌道串起來,做一個完整的影像分類專案。